In [1]:
# Windows + Jupyter compatibility — safe to keep even though Analyst doesn't use MCP
import asyncio, sys
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

import os
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

load_dotenv(dotenv_path='../.env')
assert os.getenv('OPENROUTER_API_KEY'), 'OPENROUTER_API_KEY missing in .env'
print('Environment loaded ✓')

Environment loaded ✓


In [2]:
llm = LLM(
    model='openrouter/openai/gpt-oss-120b:free',
    base_url='https://openrouter.ai/api/v1',
    api_key=os.getenv('OPENROUTER_API_KEY'),
)
print('LLM configured ✓')

LLM configured ✓


In [3]:
SAMPLE_REVIEW = (
    "The headphones have decent sound for the price, but the bass is "
    "muddy at high volumes. Battery life is about 6 hours, not the 10 "
    "advertised. Build feels plasticky. Would consider returning."
)
print(SAMPLE_REVIEW)

The headphones have decent sound for the price, but the bass is muddy at high volumes. Battery life is about 6 hours, not the 10 advertised. Build feels plasticky. Would consider returning.


In [4]:
analyst = Agent(
    role='Senior Review Analyst',
    goal=(
        'Break a review into structured insights: topic, praises, '
        'complaints, emotional tone, and factual claims.'
    ),
    backstory=(
        'Ten years of e-commerce consumer-insights experience. '
        'You read between the lines and separate opinion from fact.'
    ),
    llm=llm,
    verbose=True,
)

analyze_task = Task(
    description=(
        'Analyze the following customer review and produce a structured '
        'report with these five sections:\n'
        '1. Main topic / product feature\n'
        '2. Specific praises (bullet list)\n'
        '3. Specific complaints (bullet list)\n'
        '4. Emotional tone (e.g. excited, frustrated, indifferent)\n'
        '5. Any factual claims worth verifying\n\n'
        'REVIEW:\n{review_text}'
    ),
    expected_output='A clear, bulleted analysis covering all five sections.',
    agent=analyst,
)

crew = Crew(
    agents=[analyst],
    tasks=[analyze_task],
    process=Process.sequential,
    verbose=True,
)

result = crew.kickoff(inputs={'review_text': SAMPLE_REVIEW})

print('\n=========== ANALYST OUTPUT ===========')
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 8a9a2101-6012-4e05-98f6-64ffdd8064e6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the following customer review and produce a structured report with these five sections:          │
│  1. Main topic / product feature                                                                                │
│  2. Specific praises (bullet list)                                                                              │
│  3. Specific complaints (bullet list)                                                                           │
│  4. Emotional tone (e.g. excited, frustrated, indifferent)                                                      │
│  5. Any factual claims worth verifying                                                                          │
│                                                                                                                 │
│  REVIEW:                                                                                                        │
│  The headphones have decent sound for the price, but the bass is muddy at high volumes. Battery life is about   │
│  6 hours, not the 10 advertised. Build feels plasticky. Would consider returning.                               │
│  ID: 0fd747a6-ccba-411a-a0e4-cac1e824c4bc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Review Analyst                                                                                   │
│                                                                                                                 │
│  Task: Analyze the following customer review and produce a structured report with these five sections:          │
│  1. Main topic / product feature                                                                                │
│  2. Specific praises (bullet list)                                                                              │
│  3. Specific complaints (bullet list)                                                                           │
│  4. Emotional tone (e.g. excited, frustrated, indifferent)                                                      │
│  5. Any factual claims worth verifying                                                                          │
│                                                                                                                 │
│  REVIEW:                                                                                                        │
│  The headphones have decent sound for the price, but the bass is muddy at high volumes. Battery life is about   │
│  6 hours, not the 10 advertised. Build feels plasticky. Would consider returning.                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Review Analyst                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **1. Main topic / product feature**                                                                            │
│  - Wireless headphones (focus on sound quality, bass performance, battery life, and build quality).             │
│                                                                                                                 │
│  **2. Specific praises**                                                                                        │
│  - “Decent sound for the price.”                                                                                │
│  - Implicit satisfaction with overall value at the price point.                                                 │
│                                                                                                                 │
│  **3. Specific complaints**                                                                                     │
│  - “Bass is muddy at high volumes.”                                                                             │
│  - “Battery life is about 6 hours, not the 10 advertised.”                                                      │
│  - “Build feels plasticky.”                                                                                     │
│  - “Would consider returning.” (indicates potential dissatisfaction strong enough to contemplate a return).     │
│                                                                                                                 │
│  **4. Emotional tone**                                                                                          │
│  - Mildly frustrated/disappointed (recognizes fair value but notes several shortcomings and is considering      │
│  return).                                                                                                       │
│                                                                                                                 │
│  **5. Factual claims worth verifying**                                                                          │
│  - Battery life claim: actual runtime ~6 hours vs. manufacturer’s advertised 10 hours.                          │
│  - Bass quality description (“muddy at high volumes”) – could be tested against frequency response specs.       │
│  - Build material description (“plasticky”) – could be checked against product specifications/materials list.   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Analyze the following customer review and produce a structured report with these five sections:          │
│  1. Main topic / product feature                                                                                │
│  2. Specific praises (bullet list)                                                                              │
│  3. Specific complaints (bullet list)                                                                           │
│  4. Emotional tone (e.g. excited, frustrated, indifferent)                                                      │
│  5. Any factual claims worth verifying                                                                          │
│                                                                                                                 │
│  REVIEW:                                                                                                        │
│  The headphones have decent sound for the price, but the bass is muddy at high volumes. Battery life is about   │
│  6 hours, not the 10 advertised. Build feels plasticky. Would consider returning.                               │
│  Agent: Senior Review Analyst                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 8a9a2101-6012-4e05-98f6-64ffdd8064e6                                                                       │
│  Final Output: **1. Main topic / product feature**                                                              │
│  - Wireless headphones (focus on sound quality, bass performance, battery life, and build quality).             │
│                                                                                                                 │
│  **2. Specific praises**                                                                                        │
│  - “Decent sound for the price.”                                                                                │
│  - Implicit satisfaction with overall value at the price point.                                                 │
│                                                                                                                 │
│  **3. Specific complaints**                                                                                     │
│  - “Bass is muddy at high volumes.”                                                                             │
│  - “Battery life is about 6 hours, not the 10 advertised.”                                                      │
│  - “Build feels plasticky.”                                                                                     │
│  - “Would consider returning.” (indicates potential dissatisfaction strong enough to contemplate a return).     │
│                                                                                                                 │
│  **4. Emotional tone**                                                                                          │
│  - Mildly frustrated/disappointed (recognizes fair value but notes several shortcomings and is considering      │
│  return).                                                                                                       │
│                                                                                                                 │
│  **5. Factual claims worth verifying**                                                                          │
│  - Battery life claim: actual runtime ~6 hours vs. manufacturer’s advertised 10 hours.                          │
│  - Bass quality description (“muddy at high volumes”) – could be tested against frequency response specs.       │
│  - Build material description (“plasticky”) – could be checked against product specifications/materials list.   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=========== ANALYST OUTPUT ===========
**1. Main topic / product feature**  
- Wireless headphones (focus on sound quality, bass performance, battery life, and build quality).

**2. Specific praises**  
- “Decent sound for the price.”  
- Implicit satisfaction with overall value at the price point.

**3. Specific complaints**  
- “Bass is muddy at high volumes.”  
- “Battery life is about 6 hours, not the 10 advertised.”  
- “Build feels plasticky.”  
- “Would consider returning.” (indicates potential dissatisfaction strong enough to contemplate a return).

**4. Emotional tone**  
- Mildly frustrated/disappointed (recognizes fair value but notes several shortcomings and is considering return).

**5. Factual claims worth verifying**  
- Battery life claim: actual runtime ~6 hours vs. manufacturer’s advertised 10 hours.  
- Bass quality description (“muddy at high volumes”) – could be tested against frequency response specs.  
- Build material description (“plasticky”) – could be checke

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯